# LeetCode 4091 — Count Magical Substring Pairs II (Hard)

Bài toán LeetCode:

```python
class Solution:
    def countMagicalPairs(self, s: str) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4091 (Biweekly Contest 194 — câu 4, rating khoảng 2155).

Ta phải đếm các **cặp đoạn con kề nhau** có cùng số ký tự phân biệt. Có tới `O(n³)` bộ ba `(i, j, k)` — nhưng ý tưởng then chốt rất đẹp:

> **Cố định chỗ cắt `j`.** Đoạn trái và đoạn phải **độc lập** với nhau. Chỉ cần biết *"có bao nhiêu đoạn kết thúc tại `j` có đúng `d` ký tự phân biệt"* và *"có bao nhiêu đoạn bắt đầu tại `j + 1` có đúng `d` ký tự phân biệt"*, rồi nhân lại.

Vì bảng chữ cái chỉ có 26 chữ, `d` chỉ từ 1 tới 26. Ta sẽ học hai cách tính các con số đó trong `O(26 · n)`.

---

## Sau notebook này, bạn sẽ nắm được

1. Đếm số ký tự phân biệt tăng dần (incremental) bằng `set` / mảng đếm.
2. Brute Force `O(n³)` và cải tiến `O(n²)` bằng cách tách theo chỗ cắt.
3. Kỹ thuật **"vị trí xuất hiện gần nhất của từng chữ cái"** để đếm đoạn kết thúc tại `j` có đúng `d` ký tự phân biệt.
4. Kỹ thuật **"đúng `k` = tối đa `k` − tối đa `k−1`"** với cửa sổ trượt.
5. Mẹo **đảo ngược chuỗi** để chuyển "đoạn bắt đầu tại" thành "đoạn kết thúc tại".
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho chuỗi chữ thường `s` độ dài `n`. Đếm số bộ ba `(i, j, k)` với `0 <= i <= j < k < n` sao cho:

```text
s1 = s[i..j]   và   s2 = s[j+1..k]
có CÙNG số ký tự phân biệt
```

(`s1` và `s2` là hai đoạn con **kề nhau**: `s1` kết thúc ngay trước khi `s2` bắt đầu.)

---

## Ví dụ 1

```python
s = "aba"   ->   2
```

- `(0, 0, 1)`: `"a"` | `"b"` — 1 và 1 ✔
- `(1, 1, 2)`: `"b"` | `"a"` — 1 và 1 ✔
- `(0, 0, 2)`: `"a"` | `"ba"` — 1 và 2 ✗; `(0, 1, 2)`: `"ab"` | `"a"` — 2 và 1 ✗.

## Ví dụ 2

```python
s = "kkk"   ->   4
```

Mọi đoạn đều có 1 ký tự phân biệt → mọi bộ ba đều hợp lệ: `(0,0,1)`, `(0,0,2)`, `(0,1,2)`, `(1,1,2)`.

## Ví dụ 3

```python
s = "abcac"   ->   7
```

4 cặp hai ký tự đơn liền nhau, cộng thêm `"ab"|"ca"`, `"ab"|"cac"`, `"bc"|"ac"` (đều 2 và 2).

---

## Ràng buộc

```text
2 <= s.length <= 10^5
s chỉ gồm chữ cái tiếng Anh viết thường
```

# 2. Kiến thức nền tảng

## 2.1. Đếm số ký tự phân biệt tăng dần

Khi kéo dài một đoạn thêm một ký tự, số ký tự phân biệt tăng 1 nếu ký tự đó **chưa có**, ngược lại giữ nguyên. Dùng `set` để kiểm tra trong `O(1)`.

In [1]:
s = "abcac"
seen = set()
for j, ch in enumerate(s):
    seen.add(ch)
    print(f"s[0..{j}] = {s[:j + 1]!r:8} -> {len(seen)} ký tự phân biệt")

s[0..0] = 'a'      -> 1 ký tự phân biệt
s[0..1] = 'ab'     -> 2 ký tự phân biệt
s[0..2] = 'abc'    -> 3 ký tự phân biệt
s[0..3] = 'abca'   -> 3 ký tự phân biệt
s[0..4] = 'abcac'  -> 3 ký tự phân biệt


## 2.2. Tách theo chỗ cắt: phép nhân

Với chỗ cắt cố định `j` (`s1` kết thúc ở `j`, `s2` bắt đầu ở `j + 1`):

- gọi `left[d]` = số đoạn **kết thúc tại `j`** có đúng `d` ký tự phân biệt (chọn `i`);
- gọi `right[d]` = số đoạn **bắt đầu tại `j + 1`** có đúng `d` ký tự phân biệt (chọn `k`).

Việc chọn `i` và chọn `k` **độc lập**, nên số bộ ba hợp lệ với chỗ cắt `j` là:

```text
Σ_d  left[d] · right[d]
```

Đây là **quy tắc nhân** trong tổ hợp: chọn A có `a` cách, chọn B có `b` cách, chọn cả hai có `a · b` cách.

In [2]:
# Ví dụ: s = "abcac", chỗ cắt j = 1  (s1 kết thúc ở chỉ số 1, s2 bắt đầu ở chỉ số 2)
s = "abcac"
j = 1
left = {}
for i in range(j, -1, -1):
    d = len(set(s[i:j + 1]))
    left[d] = left.get(d, 0) + 1
right = {}
for k in range(j + 1, len(s)):
    d = len(set(s[j + 1:k + 1]))
    right[d] = right.get(d, 0) + 1
print("left  =", left, "(đoạn kết thúc tại 1: 'b', 'ab')")
print("right =", right, "(đoạn bắt đầu tại 2: 'c', 'ca', 'cac')")
print("số bộ ba =", sum(left[d] * right.get(d, 0) for d in left))

left  = {1: 1, 2: 1} (đoạn kết thúc tại 1: 'b', 'ab')
right = {1: 1, 2: 2} (đoạn bắt đầu tại 2: 'c', 'ca', 'cac')
số bộ ba = 3


# 3. Cách 1 — Brute Force O(n³)

Duyệt `i`, `j` (đếm phân biệt tăng dần), rồi `k` (đếm phân biệt tăng dần).

In [3]:
def count_bruteforce(s: str) -> int:
    n = len(s)
    total = 0
    for i in range(n):
        left_set = set()
        for j in range(i, n - 1):
            left_set.add(s[j])
            d1 = len(left_set)
            right_set = set()
            for k in range(j + 1, n):
                right_set.add(s[k])
                if len(right_set) == d1:
                    total += 1
    return total


print(count_bruteforce("aba"))     # 2
print(count_bruteforce("kkk"))     # 4
print(count_bruteforce("abcac"))   # 7

2
4
7


`O(n³)` — với `n = 10^5` là không thể.

# 4. Cách 2 — Tách theo chỗ cắt: O(n²)

Áp dụng mục 2.2: với mỗi `j`, tính `left[1..26]` (quét `i` từ `j` về `0`) và `right[1..26]` (quét `k` từ `j+1` tới cuối), rồi nhân.

In [4]:
def count_split(s: str) -> int:
    n = len(s)
    total = 0
    for j in range(n - 1):
        left = [0] * 27
        seen = set()
        for i in range(j, -1, -1):
            seen.add(s[i])
            left[len(seen)] += 1
        right = [0] * 27
        seen = set()
        for k in range(j + 1, n):
            seen.add(s[k])
            right[len(seen)] += 1
        total += sum(left[d] * right[d] for d in range(1, 27))
    return total


print(count_split("aba"), count_split("kkk"), count_split("abcac"))

2 4 7


Vẫn `O(n²)`. Cần tính `left[·]` cho **mọi** `j` nhanh hơn: mỗi `j` chỉ `O(26)`.

---

# 5. Bước đột phá: vị trí xuất hiện gần nhất của từng chữ cái

Cố định điểm cuối `j`. Với mỗi chữ cái `c` đã xuất hiện trong `s[0..j]`, gọi `last[c]` = vị trí xuất hiện **gần nhất** (`<= j`). Sắp các vị trí này **giảm dần**:

```text
p1 > p2 > p3 > ...        (p1 = j vì s[j] vừa xuất hiện)
```

Đoạn `s[i..j]` chứa chữ cái `c` ⇔ `i <= last[c]`. Vậy khi `i` lùi dần từ `j`:

- `i ∈ (p2, p1]` → đoạn chứa đúng **1** chữ cái;
- `i ∈ (p3, p2]` → đúng **2** chữ cái;
- ...
- `i ∈ (p_{d+1}, p_d]` → đúng **d** chữ cái (với `p_{d+1} = -1` nếu không có).

Do đó:

```text
left_j[d] = p_d - p_{d+1}
```

## Ví dụ `s = "abcac"`, `j = 4`

`last = {a: 3, b: 1, c: 4}` → sắp giảm dần `p = [4, 3, 1]`, thêm `p4 = -1`.

| d | khoảng i | số đoạn | các đoạn |
|---:|---|---:|---|
| 1 | `(3, 4]` | 1 | `"c"` |
| 2 | `(1, 3]` | 2 | `"ac"`, `"cac"` |
| 3 | `(-1, 1]` | 2 | `"bcac"`, `"abcac"` |

## Cập nhật khi tăng `j`: "chuyển lên đầu" (move-to-front)

Giữ danh sách `order` các chữ cái sắp theo `last` giảm dần. Khi gặp `s[j]`: xóa nó khỏi vị trí cũ (nếu có) và **chèn lên đầu**. Danh sách dài tối đa 26 → mỗi bước `O(26)`.

In [5]:
def ending_counts(s: str):
    # res[j][d] = số đoạn kết thúc tại j có đúng d ký tự phân biệt (d = 1..26)
    order = []                  # các chữ cái, sắp theo vị trí gần nhất giảm dần
    last = {}
    res = []
    for j, ch in enumerate(s):
        if ch in last:
            order.remove(ch)
        order.insert(0, ch)     # chuyển lên đầu
        last[ch] = j
        cnt = [0] * 27
        for d in range(1, len(order) + 1):
            hi = last[order[d - 1]]
            lo = last[order[d]] if d < len(order) else -1
            cnt[d] = hi - lo
        res.append(cnt)
    return res


L = ending_counts("abcac")
for j, row in enumerate(L):
    print(f"j = {j} ({'abcac'[j]}): đúng 1/2/3 ký tự phân biệt -> {row[1:4]}")

j = 0 (a): đúng 1/2/3 ký tự phân biệt -> [1, 0, 0]
j = 1 (b): đúng 1/2/3 ký tự phân biệt -> [1, 1, 0]
j = 2 (c): đúng 1/2/3 ký tự phân biệt -> [1, 1, 1]
j = 3 (a): đúng 1/2/3 ký tự phân biệt -> [1, 1, 2]
j = 4 (c): đúng 1/2/3 ký tự phân biệt -> [1, 2, 2]


## Đoạn **bắt đầu** tại một vị trí: đảo ngược chuỗi

Đoạn `s[j+1..k]` trong `s` chính là một đoạn **kết thúc** tại vị trí tương ứng trong chuỗi đảo ngược `s[::-1]`. Vị trí `t` của `s` ứng với vị trí `n - 1 - t` của chuỗi đảo. Vậy:

```text
R = ending_counts(s[::-1])[::-1]
R[t][d] = số đoạn BẮT ĐẦU tại t (trong s) có đúng d ký tự phân biệt
```

In [6]:
R = ending_counts("abcac"[::-1])[::-1]
for t, row in enumerate(R):
    print(f"t = {t}: đoạn bắt đầu tại {t} có 1/2/3 ký tự phân biệt -> {row[1:4]}")

t = 0: đoạn bắt đầu tại 0 có 1/2/3 ký tự phân biệt -> [1, 1, 3]
t = 1: đoạn bắt đầu tại 1 có 1/2/3 ký tự phân biệt -> [1, 1, 2]
t = 2: đoạn bắt đầu tại 2 có 1/2/3 ký tự phân biệt -> [1, 2, 0]
t = 3: đoạn bắt đầu tại 3 có 1/2/3 ký tự phân biệt -> [1, 1, 0]
t = 4: đoạn bắt đầu tại 4 có 1/2/3 ký tự phân biệt -> [1, 0, 0]


# 6. Cách 3 — Vị trí gần nhất + đảo chuỗi: O(26 · n)

```text
answer = Σ_{j=0}^{n-2}  Σ_{d=1}^{26}  L[j][d] · R[j+1][d]
```

## Mô phỏng `s = "abcac"`

| chỗ cắt j | `L[j][1..3]` | `R[j+1][1..3]` | tích vô hướng |
|---:|---|---|---:|
| 0 | `[1, 0, 0]` | `[1, 1, 2]` | 1 |
| 1 | `[1, 1, 0]` | `[1, 2, 0]` | `1 + 2 = 3` |
| 2 | `[1, 1, 1]` | `[1, 1, 0]` | `1 + 1 = 2` |
| 3 | `[1, 1, 2]` | `[1, 0, 0]` | 1 |

Tổng `7` ✔.

In [7]:
def count_last_positions(s: str) -> int:
    n = len(s)
    L = ending_counts(s)
    R = ending_counts(s[::-1])[::-1]
    return sum(L[j][d] * R[j + 1][d] for j in range(n - 1) for d in range(1, 27))


def trace_last_positions(s: str) -> int:
    n = len(s)
    L = ending_counts(s)
    R = ending_counts(s[::-1])[::-1]
    total = 0
    for j in range(n - 1):
        dot = sum(L[j][d] * R[j + 1][d] for d in range(1, 27))
        total += dot
        print(f"j = {j}: L[j] = {L[j][1:5]}, R[j+1] = {R[j + 1][1:5]} -> +{dot}, tổng = {total}")
    return total


print(count_last_positions("aba"), count_last_positions("kkk"), count_last_positions("abcac"))
trace_last_positions("abcac");

2 4 7
j = 0: L[j] = [1, 0, 0, 0], R[j+1] = [1, 1, 2, 0] -> +1, tổng = 1
j = 1: L[j] = [1, 1, 0, 0], R[j+1] = [1, 2, 0, 0] -> +3, tổng = 4
j = 2: L[j] = [1, 1, 1, 0], R[j+1] = [1, 1, 0, 0] -> +2, tổng = 6
j = 3: L[j] = [1, 1, 2, 0], R[j+1] = [1, 0, 0, 0] -> +1, tổng = 7


# 7. Cách 4 — "Đúng `k` = tối đa `k` − tối đa `k−1`" bằng cửa sổ trượt

Một cách tính `L[j][k]` khác, rất hay gặp trong các bài đếm đoạn con:

- Gọi `f_k(j)` = độ dài đoạn **dài nhất** kết thúc tại `j` có **tối đa** `k` ký tự phân biệt. Mọi điểm đầu `i` trong `[j - f_k(j) + 1, j]` đều cho đoạn có `<= k` ký tự phân biệt (bớt ký tự thì số phân biệt không tăng).
- Vậy số đoạn kết thúc tại `j` có **tối đa** `k` ký tự phân biệt là `f_k(j)`, và số đoạn có **đúng** `k` là `f_k(j) - f_{k-1}(j)`.

Với mỗi `k = 1..26`, tính `f_k(j)` cho mọi `j` bằng **hai con trỏ** (cửa sổ trượt "tối đa `k` ký tự phân biệt") trong `O(n)`.

In [8]:
def at_most_lengths(s: str, k: int):
    # f[j] = độ dài dài nhất của đoạn kết thúc tại j có <= k ký tự phân biệt
    cnt = {}
    left = 0
    f = []
    for right, ch in enumerate(s):
        cnt[ch] = cnt.get(ch, 0) + 1
        while len(cnt) > k:
            c = s[left]
            cnt[c] -= 1
            if cnt[c] == 0:
                del cnt[c]
            left += 1
        f.append(right - left + 1)
    return f


def ending_counts_window(s: str):
    n = len(s)
    F = [[0] * n] + [at_most_lengths(s, k) for k in range(1, 27)]
    return [[0] + [F[k][j] - F[k - 1][j] for k in range(1, 27)] for j in range(n)]


def count_window(s: str) -> int:
    n = len(s)
    L = ending_counts_window(s)
    R = ending_counts_window(s[::-1])[::-1]
    return sum(L[j][d] * R[j + 1][d] for j in range(n - 1) for d in range(1, 27))


print(at_most_lengths("abcac", 2))         # độ dài dài nhất với <= 2 ký tự phân biệt
print(count_window("aba"), count_window("kkk"), count_window("abcac"))

[1, 2, 2, 2, 3]
2 4 7


Cách 4 cũng là `O(26 · n)` (26 lượt hai con trỏ), dễ nghĩ ra nếu đã quen kỹ thuật "tối đa `k`".

# 8. Phiên bản đúng format LeetCode

Ta nộp cách 3. Để nhanh hơn, ta không lưu toàn bộ bảng `L` mà chỉ lưu `R` (cho các vị trí bắt đầu), rồi tính `L` "trực tiếp" khi quét từ trái sang phải.

In [9]:
class Solution:
    def countMagicalPairs(self, s: str) -> int:
        n = len(s)

        # R[t][d]: số đoạn bắt đầu tại t có đúng d ký tự phân biệt (quét từ phải sang trái)
        R = [None] * n
        order, nxt = [], {}
        for t in range(n - 1, -1, -1):
            ch = s[t]
            if ch in nxt:
                order.remove(ch)
            order.insert(0, ch)
            nxt[ch] = t
            row = [0] * (len(order) + 1)
            for d in range(1, len(order) + 1):
                lo = nxt[order[d - 1]]
                hi = nxt[order[d]] if d < len(order) else n
                row[d] = hi - lo
            R[t] = row

        total = 0
        order, last = [], {}
        for j in range(n - 1):
            ch = s[j]
            if ch in last:
                order.remove(ch)
            order.insert(0, ch)
            last[ch] = j
            right = R[j + 1]
            for d in range(1, min(len(order), len(right) - 1) + 1):
                hi = last[order[d - 1]]
                lo = last[order[d]] if d < len(order) else -1
                total += (hi - lo) * right[d]
        return total

In [10]:
sol = Solution()
tests = [
    ("aba", 2, "ví dụ 1"),
    ("kkk", 4, "ví dụ 2"),
    ("abcac", 7, "ví dụ 3"),
    ("ab", 1, "độ dài 2"),
    ("aab", 3, "bài tập 1"),
    ("abcd", 4, "toàn khác nhau"),
]
for s, expected, name in tests:
    result = sol.countMagicalPairs(s)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:16} -> {result}")

PASS | ví dụ 1          -> 2
PASS | ví dụ 2          -> 4
PASS | ví dụ 3          -> 7
PASS | độ dài 2         -> 1
PASS | bài tập 1        -> 3
PASS | toàn khác nhau   -> 4


In [11]:
import random
import time

random.seed(4091)
big = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5))
t0 = time.perf_counter()
r = sol.countMagicalPairs(big)
print(f"n = 10^5: {r:,} ({time.perf_counter() - t0:.2f}s)")

n = 10^5: 165,681,904,359,880 (0.47s)


# 9. Vì sao đúng?

1. **Phân hoạch theo chỗ cắt:** mỗi bộ ba `(i, j, k)` có đúng một chỗ cắt `j`. Với `j` cố định, các lựa chọn `i` và `k` độc lập → số bộ ba hợp lệ là `Σ_d left[d] · right[d]`.
2. **Công thức `p_d - p_{d+1}`:** đoạn `s[i..j]` chứa chữ cái `c` ⇔ `i <= last[c]`. Khi `i` giảm, các chữ cái "được thêm vào" theo thứ tự `last` giảm dần. Đoạn có đúng `d` chữ cái ⇔ `p_{d+1} < i <= p_d`.
3. **Đảo chuỗi:** đoạn bắt đầu tại `t` của `s` ↔ đoạn kết thúc tại `n - 1 - t` của `s[::-1]`, cùng tập ký tự.

---

## Trực giác

> Tưởng tượng một con dao cắt chuỗi làm hai. Bên trái có một "túi" các đoạn kết thúc tại vết cắt, phân loại theo số chữ cái khác nhau; bên phải cũng vậy. Mỗi cặp cùng loại là một đáp án.

# 10. Độ phức tạp

Gọi `n = |s|`, `Σ = 26`.

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | duyệt `i, j, k` | `O(n³)` | `O(Σ)` |
| 2. Tách chỗ cắt | quét trái/phải cho mỗi `j` | `O(n²)` | `O(Σ)` |
| 3. Vị trí gần nhất | `p_d - p_{d+1}`, move-to-front | `O(Σ · n)` | `O(Σ · n)` |
| 4. Tối đa `k` | 26 lượt hai con trỏ | `O(Σ · n)` | `O(Σ · n)` |

# 11. Đối chiếu bốn cách trên dữ liệu ngẫu nhiên

In [12]:
random.seed(2026)
for _ in range(1500):
    s = "".join(random.choice("abc") for _ in range(random.randint(2, 10)))
    expected = count_bruteforce(s)
    assert count_split(s) == expected, s
    assert count_last_positions(s) == expected, s
    assert count_window(s) == expected, s
    assert sol.countMagicalPairs(s) == expected, s
print("1500 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!")

1500 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!


# 12. Những lỗi dễ mắc

## Lỗi 1 — Cho phép `s2` rỗng hoặc chồng lên `s1`

Điều kiện là `i <= j < k`: `s1 = s[i..j]` và `s2 = s[j+1..k]` đều **khác rỗng** và **kề nhau**. Nếu duyệt `j` tới `n - 1` (không chừa chỗ cho `s2`), ta sẽ truy cập `R[n]` — vượt mảng.

## Lỗi 2 — Lệch một vị trí khi ghép `L` và `R`

Phải ghép `L[j]` với `R[j + 1]`, không phải `R[j]` (khi đó hai đoạn **dùng chung** ký tự `s[j]`).

In [13]:
def wrong_shared_char(s):
    n = len(s)
    L = ending_counts(s)
    R = ending_counts(s[::-1])[::-1]
    return sum(L[j][d] * R[j][d] for j in range(n - 1) for d in range(1, 27))   # SAI: R[j]


print("Sai :", wrong_shared_char("abcac"))
print("Đúng:", sol.countMagicalPairs("abcac"))

Sai : 8
Đúng: 7


## Lỗi 3 — Quên lấy **hiệu** `f_k - f_{k-1}` ở cách 4

`f_k(j)` đếm đoạn có **tối đa** `k` ký tự phân biệt. Ghép trực tiếp các con số "tối đa" sẽ đếm cặp `(1 ký tự, 2 ký tự)`... là sai.

In [14]:
def wrong_at_most(s):
    n = len(s)
    F = [at_most_lengths(s, k) for k in range(1, 27)]
    G = [at_most_lengths(s[::-1], k)[::-1] for k in range(1, 27)]
    return sum(F[k][j] * G[k][j + 1] for j in range(n - 1) for k in range(26))   # SAI


print("Sai :", wrong_at_most("aba"))
print("Đúng:", sol.countMagicalPairs("aba"))

Sai : 102
Đúng: 2


## Lỗi 4 — Đảo chuỗi nhưng quên đảo lại mảng kết quả

`ending_counts(s[::-1])` cho kết quả theo chỉ số của **chuỗi đảo**. Phải `[::-1]` thêm lần nữa để đưa về chỉ số của `s`.

## Lỗi 5 — Brute Force `O(n²)` với `n = 10^5`

`10^10` bước → TLE. Bảng chữ cái nhỏ (26) là gợi ý mạnh rằng lời giải có dạng `O(26 · n)`.

# 13. Mẹo nhận diện

1. Đếm **cặp đoạn kề nhau** (hoặc bộ ba chỉ số `i <= j < k`) → **cố định điểm nối `j`**, nhân số cách chọn hai bên.
2. Điều kiện chỉ phụ thuộc **số ký tự phân biệt** + bảng chữ cái nhỏ → phân loại theo `d = 1..26`.
3. "Số đoạn kết thúc tại `j` có đúng `d` ký tự phân biệt" → **vị trí xuất hiện gần nhất** (`p_d - p_{d+1}`) hoặc **tối đa `d` − tối đa `d−1`**.
4. Cần thông tin "bắt đầu tại" → **đảo chuỗi** và tái sử dụng hàm "kết thúc tại".

Các bài liên quan: *Count Magical Substring Pairs I* (4090), *Subarrays with K Different Integers* (992), *Number of Good Ways to Split a String* (1525), *Count Substrings That Differ by One Character* (1638).

# 14. Biến thể

## Biến thể 1 — Đếm cặp mà `s1` có **nhiều hơn** `s2` đúng một ký tự phân biệt

Chỉ thay tích vô hướng: `Σ_d L[j][d+1] · R[j+1][d]`.

In [15]:
def count_one_more(s: str) -> int:
    n = len(s)
    L = ending_counts(s)
    R = ending_counts(s[::-1])[::-1]
    return sum(L[j][d + 1] * R[j + 1][d] for j in range(n - 1) for d in range(1, 26))


def brute_one_more(s):
    n = len(s)
    return sum(1 for i in range(n) for j in range(i, n - 1) for k in range(j + 1, n)
               if len(set(s[i:j + 1])) == len(set(s[j + 1:k + 1])) + 1)


print(count_one_more("abcac"), brute_one_more("abcac"))

4 4


## Biến thể 2 — Số cách tách chuỗi thành hai phần có cùng số ký tự phân biệt (LeetCode 1525)

Trường hợp đặc biệt: `i = 0` và `k = n - 1` cố định. Chỉ cần đếm số phân biệt của tiền tố và hậu tố.

In [16]:
def num_good_splits(s: str) -> int:
    n = len(s)
    pre, seen = [], set()
    for ch in s:
        seen.add(ch)
        pre.append(len(seen))
    suf, seen = [0] * n, set()
    for t in range(n - 1, -1, -1):
        seen.add(s[t])
        suf[t] = len(seen)
    return sum(pre[j] == suf[j + 1] for j in range(n - 1))


print(num_good_splits("aacaba"))   # 2
print(num_good_splits("abcd"))     # 1

2
1


# 15. Template tư duy

```python
L = ending_counts(s)                 # L[j][d]: đoạn kết thúc tại j, đúng d phân biệt
R = ending_counts(s[::-1])[::-1]     # R[t][d]: đoạn bắt đầu tại t, đúng d phân biệt
answer = sum(L[j][d] * R[j + 1][d] for j in range(n - 1) for d in range(1, 27))
```

với `ending_counts` dùng danh sách "chuyển lên đầu" và công thức `p_d - p_{d+1}`.

# 16. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `s = "aab"`, lập bảng `j | L[j][1..2] | R[j+1][1..2] | tích`. Đáp án là bao nhiêu? Kiểm tra bằng cell dưới.

In [17]:
# Kiểm tra Bài 1
trace_last_positions("aab");

j = 0: L[j] = [1, 0, 0, 0], R[j+1] = [1, 1, 0, 0] -> +1, tổng = 1
j = 1: L[j] = [2, 0, 0, 0], R[j+1] = [1, 0, 0, 0] -> +2, tổng = 3


## Bài 2 — Dự đoán output

```python
print(at_most_lengths("aabbc", 1))
print(at_most_lengths("aabbc", 2))
print(ending_counts("aba")[2][1:4])
```

In [18]:
print(at_most_lengths("aabbc", 1))
print(at_most_lengths("aabbc", 2))
print(ending_counts("aba")[2][1:4])

[1, 2, 1, 2, 1]
[1, 2, 3, 4, 3]
[1, 2, 0]


## Bài 3 — Hoàn thiện `ending_counts`

In [19]:
def exercise_3_ending_counts(s: str):
    order, last, res = [], {}, []
    for j, ch in enumerate(s):
        # TODO 1: chuyển ch lên đầu order, cập nhật last[ch]
        # TODO 2: cnt[d] = last[order[d-1]] - (last[order[d]] nếu có, ngược lại -1)
        pass
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_ending_counts("abcac")[4][1:4] == [1, 2, 2]

## Bài 4 — Đếm đoạn con có đúng `k` ký tự phân biệt

```python
count_exactly_k("pqpqs", 2)    -> 7
count_exactly_k("aabab", 3)    -> 0
```

Gợi ý: `Σ_j (f_k(j) - f_{k-1}(j))`.

In [20]:
def exercise_4_count_exactly_k(s: str, k: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_4_count_exactly_k("pqpqs", 2) == 7
# assert exercise_4_count_exactly_k("aabab", 3) == 0

## Bài 5 — Biến thể: `s1` và `s2` cùng **độ dài** và cùng số ký tự phân biệt

Đếm bộ ba `(i, j, k)` với `j - i == k - j - 1` và cùng số ký tự phân biệt. Viết lời giải `O(n²)` (duyệt chỗ cắt và độ dài, đếm phân biệt tăng dần).

```python
count_same_length("abab")   -> 4     # "a|b", "b|a", "a|b", "ab|ab"
```

In [21]:
def exercise_5_count_same_length(s: str) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_count_same_length("abab") == 4

## Bài 6 — Mini challenge: tổng số phân biệt bằng `T`

Đếm bộ ba `(i, j, k)` sao cho `distinct(s1) + distinct(s2) == T`, trong `O(26² · n)`.

```python
count_sum_T("abcac", 3)   -> ?    (tự kiểm tra bằng brute force)
```

In [22]:
def exercise_6_count_sum_T(s: str, T: int) -> int:
    # TODO
    raise NotImplementedError


# print(exercise_6_count_sum_T("abcac", 3))

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `if ch in last: order.remove(ch)`; `order.insert(0, ch)`; `last[ch] = j`.
- Mức 2: `hi = last[order[d - 1]]`, `lo = last[order[d]] if d < len(order) else -1`.
- Mức 3: `cnt[d] = hi - lo`; `res.append(cnt)`; cuối cùng `return res`.

**Bài 4**

- Mức 1: `f_k = at_most_lengths(s, k)`, `f_{k-1} = at_most_lengths(s, k - 1)` (với `k - 1 = 0` thì toàn số 0).
- Mức 2: cộng `f_k[j] - f_{k-1}[j]` trên mọi `j`.
- Mức 3: hoặc dùng `ending_counts(s)[j][k]`.

**Bài 5**

- Mức 1: cố định chỗ cắt `j`, cho độ dài `len` tăng từ 1.
- Mức 2: `s1 = s[j-len+1..j]`, `s2 = s[j+1..j+len]` — mỗi lần tăng `len` thêm một ký tự vào mỗi bên.
- Mức 3: hai mảng đếm 26 chữ cái + hai biến số phân biệt, cập nhật `O(1)`.

**Bài 6**

- Mức 1: giống bài gốc nhưng ghép `L[j][d1]` với `R[j+1][d2]` khi `d1 + d2 == T`.
- Mức 2: `Σ_{d1} L[j][d1] · R[j+1][T - d1]` với `1 <= T - d1 <= 26`.
- Mức 3: cẩn thận chỉ số ngoài khoảng `1..26`.

# 17. Đáp án tham khảo

## Đáp án Bài 1

`s = "aab"`: `L[0] = [1, 0]`, `L[1] = [2, 0]`; `R[1] = [1, 1]`, `R[2] = [1, 0]`.

| j | `L[j][1..2]` | `R[j+1][1..2]` | tích |
|---:|---|---|---:|
| 0 | `[1, 0]` | `[1, 1]` | 1 |
| 1 | `[2, 0]` | `[1, 0]` | 2 |

Tổng `3`: `"a"|"a"`, `"a"|"b"`, `"aa"|"b"`.

## Đáp án Bài 2

```text
[1, 2, 1, 2, 1]
[1, 2, 3, 4, 3]
[1, 2, 0]
```

In [23]:
# Đáp án Bài 3
def answer_3_ending_counts(s: str):
    order, last, res = [], {}, []
    for j, ch in enumerate(s):
        if ch in last:
            order.remove(ch)
        order.insert(0, ch)
        last[ch] = j
        cnt = [0] * 27
        for d in range(1, len(order) + 1):
            hi = last[order[d - 1]]
            lo = last[order[d]] if d < len(order) else -1
            cnt[d] = hi - lo
        res.append(cnt)
    return res


assert answer_3_ending_counts("abcac")[4][1:4] == [1, 2, 2]
print("Bài 3: PASS")

Bài 3: PASS


In [24]:
# Đáp án Bài 4
def answer_4_count_exactly_k(s: str, k: int) -> int:
    fk = at_most_lengths(s, k)
    fk1 = at_most_lengths(s, k - 1) if k > 1 else [0] * len(s)
    return sum(a - b for a, b in zip(fk, fk1))


assert answer_4_count_exactly_k("pqpqs", 2) == 7
assert answer_4_count_exactly_k("aabab", 3) == 0
print("Bài 4: PASS")

Bài 4: PASS


In [25]:
# Đáp án Bài 5
def answer_5_count_same_length(s: str) -> int:
    n = len(s)
    total = 0
    for j in range(n - 1):
        cl, cr = [0] * 26, [0] * 26
        dl = dr = 0
        length = 1
        while j - length + 1 >= 0 and j + length < n:
            a = ord(s[j - length + 1]) - 97
            b = ord(s[j + length]) - 97
            cl[a] += 1
            dl += cl[a] == 1
            cr[b] += 1
            dr += cr[b] == 1
            total += dl == dr
            length += 1
    return total


def brute_same_length(s):
    n = len(s)
    return sum(1 for i in range(n) for j in range(i, n - 1) for k in range(j + 1, n)
               if j - i == k - j - 1 and len(set(s[i:j + 1])) == len(set(s[j + 1:k + 1])))


assert answer_5_count_same_length("abab") == 4
random.seed(5)
for _ in range(500):
    t = "".join(random.choice("abc") for _ in range(random.randint(2, 10)))
    assert answer_5_count_same_length(t) == brute_same_length(t)
print("Bài 5: PASS (kể cả 500 test ngẫu nhiên)")

Bài 5: PASS (kể cả 500 test ngẫu nhiên)


In [26]:
# Đáp án Bài 6
def answer_6_count_sum_T(s: str, T: int) -> int:
    n = len(s)
    L = ending_counts(s)
    R = ending_counts(s[::-1])[::-1]
    total = 0
    for j in range(n - 1):
        for d1 in range(1, 27):
            d2 = T - d1
            if 1 <= d2 <= 26:
                total += L[j][d1] * R[j + 1][d2]
    return total


def brute_sum_T(s, T):
    n = len(s)
    return sum(1 for i in range(n) for j in range(i, n - 1) for k in range(j + 1, n)
               if len(set(s[i:j + 1])) + len(set(s[j + 1:k + 1])) == T)


print("abcac, T = 3 ->", answer_6_count_sum_T("abcac", 3))
random.seed(6)
for _ in range(300):
    t = "".join(random.choice("abc") for _ in range(random.randint(2, 9)))
    T = random.randint(2, 6)
    assert answer_6_count_sum_T(t, T) == brute_sum_T(t, T)
print("Bài 6: PASS (khớp brute force trên 300 test)")

abcac, T = 3 -> 7
Bài 6: PASS (khớp brute force trên 300 test)


# 18. Tổng kết

**Count Magical Substring Pairs II**:

1. **Cố định chỗ cắt** → tách bài toán thành hai bài độc lập, ghép bằng **quy tắc nhân**.
2. "Đoạn kết thúc tại `j` có đúng `d` ký tự phân biệt" = `p_d - p_{d+1}` (vị trí gần nhất, sắp giảm dần) — hoặc "tối đa `d` − tối đa `d−1`".
3. "Đoạn bắt đầu tại" = **đảo chuỗi**.

## Công thức nhớ nhanh

```text
answer = Σ_j Σ_d L[j][d] · R[j+1][d]
L[j][d] = p_d - p_{d+1}   (p: vị trí gần nhất của các chữ cái, giảm dần, p_{k+1} = -1)
```

## Độ phức tạp

```text
Time:  O(26 · n)
Space: O(26 · n)
```

# 19. Checklist trước khi submit

- [ ] Ghép `L[j]` với `R[j + 1]` (hai đoạn không dùng chung ký tự)?
- [ ] Chỗ cắt `j` chạy tới `n - 2`?
- [ ] Đảo chuỗi rồi **đảo lại** mảng kết quả?
- [ ] Nếu dùng "tối đa `k`": đã lấy hiệu `f_k - f_{k-1}`?
- [ ] Đã test: chuỗi toàn một chữ, toàn khác nhau, độ dài 2?